In [ ]:
import pandas as pd

train_data = pd.read_csv("train.csv")
test_data = pd.read_csv("test.csv")

In [ ]:
data_dictionary = {
    "row_id": "identifier",
    "credit_limit": "predictor",
    "sex": "predictor / audit attribute",
    "education": "predictor",
    "marital_status": "predictor",
    "age": "predictor",
    "latest_repayment_status": "predictor",
    "maximum_repayment_delay_6m": "predictor",
    "delayed_months_6m": "predictor",
    "average_bill_6m": "predictor",
    "bill_change_sep_to_apr": "predictor",
    "total_payment_6m": "predictor",
    "payment_to_bill_ratio_6m": "predictor",
    "zero_payment_months_6m": "predictor",
    "utilization_sep": "predictor",
    "repayment_assistance_plan": "predictor",
    "payment_difficulty_next_month": "target",
    "predicted_class": "submission"
}

categorical_columns = [
    "sex",
    "education",
    "marital_status",
    "repayment_assistance_plan",
]

In [ ]:
from sklearn.model_selection import train_test_split

predictor_columns = [
    column for column, role in data_dictionary.items() if "predictor" in role
]
target_column = [
    column for column, role in data_dictionary.items() if role == "target"
][0]

# Split only the labeled data. Keep the unlabeled test data untouched for final predictions.
X = train_data[predictor_columns]
y = train_data[target_column]

X_train, X_validation, y_train, y_validation = train_test_split(
    X,
    y,
    test_size=0.3,
    random_state=411,
    stratify=y,
)

X_test = test_data[predictor_columns]

In [ ]:
# Transform categorical variables into one-hot encoded features

from sklearn.preprocessing import OneHotEncoder

encoder = OneHotEncoder(
    handle_unknown="ignore",
    sparse_output=False
)

X_train_cat = encoder.fit_transform(X_train[categorical_columns])
X_validation_cat = encoder.transform(X_validation[categorical_columns])
X_test_cat = encoder.transform(X_test[categorical_columns])

numerical_columns = [
    col for col in predictor_columns
    if col not in categorical_columns
]

X_train_num = X_train[numerical_columns]
X_validation_num = X_validation[numerical_columns]
X_test_num = X_test[numerical_columns]

In [ ]:
# Combine the one-hot encoded categorical features and the numerical features into a single feature matrix

encoded_categorical_columns = encoder.get_feature_names_out(categorical_columns)

X_train_cat = pd.DataFrame(
    X_train_cat,
    columns=encoded_categorical_columns,
    index=X_train.index
)

X_validation_cat = pd.DataFrame(
    X_validation_cat,
    columns=encoded_categorical_columns,
    index=X_validation.index
)

X_test_cat = pd.DataFrame(
    X_test_cat,
    columns=encoded_categorical_columns,
    index=X_test.index
)

X_train_encoded = pd.concat(
    [X_train_num, X_train_cat],
    axis=1
)

X_validation_encoded = pd.concat(
    [X_validation_num, X_validation_cat],
    axis=1
)

X_test_encoded = pd.concat(
    [X_test_num, X_test_cat],
    axis=1
)

X_train_encoded.head()

In [ ]:
from interpret import show
from interpret.data import ClassHistogram

hist = ClassHistogram().explain_data(X_train, y_train, name='Train Data')
show(hist)

In [ ]:
from interpret.glassbox import ClassificationTree
from sklearn.metrics import accuracy_score, balanced_accuracy_score, classification_report

tree = ClassificationTree()
tree.fit(X_train_encoded, y_train)

validation_predictions = tree.predict(X_validation_encoded)
print(f"Validation accuracy: {accuracy_score(y_validation, validation_predictions):.3f}")
print(
    f"Validation balanced accuracy: "
    f"{balanced_accuracy_score(y_validation, validation_predictions):.3f}"
)
print(classification_report(y_validation, validation_predictions))

In [ ]:
from interpret import show
tree_global = tree.explain_global(name='Tree')
show(tree_global)

In [ ]:
# Classify the unlabeled test data and create a prediction file

import matplotlib.pyplot as plt

test_predictions = tree.predict(X_test_encoded)

# Keep the original test predictors alongside each prediction for debugging.
submission = test_data.copy()
submission["predicted_class"] = test_predictions
submission.to_csv("test_predictions.csv", index=False)

class_proportions = (
    submission["predicted_class"]
    .value_counts(normalize=True)
    .sort_index()
)

ax = class_proportions.plot(
    kind="bar",
    color=["#2563eb", "#f97316"],
    title="Proportion of predicted classes in test data",
    ylabel="Proportion",
    xlabel="Predicted class",
    ylim=(0, 1),
)
ax.set_xticklabels(class_proportions.index, rotation=0)
ax.yaxis.set_major_formatter(
    plt.FuncFormatter(lambda value, _: f"{value:.0%}")
)

for index, proportion in enumerate(class_proportions):
    ax.text(index, proportion + 0.02, f"{proportion:.1%}", ha="center")

plt.tight_layout()
plt.show()

submission.head()